# AIOps Bronze record scoring

Scores Bronze records with the trained record-level PyTorch autoencoder and writes PASS/WARN/QUARANTINE outputs.

In [ ]:
%pip install torch --quiet

In [ ]:
LAYER = "bronze"
SOURCE_TABLE_NAME = "aiops_bronze_vehicle_positions_records"
DECISION_TABLE_NAME = "report_aiops_bronze_record_decisions"
SCORE_TABLE_NAME = "report_aiops_bronze_record_scores"
VALIDATED_TABLE_NAME = "aiops_bronze_vehicle_positions_validated_stream"
WARNING_TABLE_NAME = "aiops_bronze_vehicle_positions_warning_stream"
QUARANTINE_TABLE_NAME = "aiops_bronze_vehicle_positions_quarantine_stream"
QUERY_NAME = "aiops_bronze_record_score"

In [ ]:
import base64
from functools import reduce
from io import BytesIO
import json
import time

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

try:
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
except ImportError as exc:
    raise ImportError("Use a Databricks ML Runtime or install torch before running this notebook.") from exc

CATALOG_NAME = "hant-catalog"
SCHEMA_NAME = "hsl"
STORAGE_ACCOUNT = "streanmingdatasta"
LAKEHOUSE_CONTAINER = "lakehouse"
BASE_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/external/hant-catalog"
MODEL_BASE_PATH = f"{BASE_PATH}/aiops/models"
AIOPS_BASE_PATH = f"{BASE_PATH}/aiops"

MODEL_ARTIFACTS_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.report_aiops_model_artifacts"
RUNTIME_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.report_aiops_runtime_metrics"
RUNTIME_PATH = f"{AIOPS_BASE_PATH}/runtime/report_aiops_runtime_metrics"

dbutils.widgets.text("model_version", "")
dbutils.widgets.text("warning_multiplier", "1.0")
dbutils.widgets.text("quarantine_multiplier", "3.0")
dbutils.widgets.text("top_n_features", "10")
dbutils.widgets.dropdown("persist_reports", "true", ["true", "false"])
dbutils.widgets.text("trigger_interval", "10 seconds")
dbutils.widgets.text("max_files_per_trigger", "5")
dbutils.widgets.text("max_bytes_per_trigger", "256m")
dbutils.widgets.text("max_driver_records_per_batch", "50000")
dbutils.widgets.text("scoring_chunk_records", "10000")

MODEL_VERSION_WIDGET = dbutils.widgets.get("model_version").strip()
WARNING_MULTIPLIER = float(dbutils.widgets.get("warning_multiplier"))
QUARANTINE_MULTIPLIER = float(dbutils.widgets.get("quarantine_multiplier"))
TOP_N_FEATURES = int(dbutils.widgets.get("top_n_features"))
PERSIST_REPORTS = dbutils.widgets.get("persist_reports").lower() == "true"
TRIGGER_INTERVAL = dbutils.widgets.get("trigger_interval")
MAX_FILES_PER_TRIGGER = int(dbutils.widgets.get("max_files_per_trigger"))
MAX_BYTES_PER_TRIGGER = dbutils.widgets.get("max_bytes_per_trigger").strip()
MAX_DRIVER_RECORDS_PER_BATCH = int(dbutils.widgets.get("max_driver_records_per_batch"))
SCORING_CHUNK_RECORDS = max(1, int(dbutils.widgets.get("scoring_chunk_records")))


def filter_to_target_ingest_date(df: DataFrame) -> DataFrame:
    if "ingest_date" not in df.columns:
        raise ValueError("Expected ingest_date column before AIOps scoring date filtering.")
    return df.where(F.to_date(F.col("ingest_date")) == F.current_date())


def qname(table_name: str) -> str:
    return f"`{CATALOG_NAME}`.{SCHEMA_NAME}.{table_name}"


def flag(condition):
    return F.when(condition, F.lit(1.0)).otherwise(F.lit(0.0))


def clipped_double(col_name: str, min_value: float, max_value: float):
    return (
        F.when(F.col(col_name).isNull(), F.lit(0.0))
        .when(F.col(col_name).cast("double") < F.lit(min_value), F.lit(min_value))
        .when(F.col(col_name).cast("double") > F.lit(max_value), F.lit(max_value))
        .otherwise(F.col(col_name).cast("double"))
    )


def source_type_for_rule(rule_id: str) -> str:
    r = rule_id.lower()
    if any(x in r for x in ["parse", "json", "mqtt", "event_type", "transport_mode", "format"]):
        return "schema_parse"
    if any(x in r for x in ["null", "missing", "present"]):
        return "completeness"
    if any(x in r for x in ["out_of_bounds", "negative", "range", "invalid", "unexpected"]):
        return "validity"
    if any(x in r for x in ["mismatch", "duplicate", "unique"]):
        return "consistency"
    if any(x in r for x in ["future", "stale", "timestamp", "unix", "delay"]):
        return "timeliness"
    return "other"


def feature_source_type(feature_name: str) -> str:
    if feature_name.startswith("rule_") and feature_name.endswith("_flag"):
        return source_type_for_rule(feature_name.removeprefix("rule_").removesuffix("_flag"))
    f = feature_name.lower()
    if any(x in f for x in ["lat", "lon", "speed", "heading", "occupancy", "hour"]):
        return "validity"
    if any(x in f for x in ["delay", "stale", "future", "enqueue", "ingest"]):
        return "timeliness"
    if any(x in f for x in ["duplicate", "business_key"]):
        return "consistency"
    return "other"


def rule_metadata(rule_specs: list[dict]) -> str:
    return json.dumps([
        {
            "rule_id": spec["rule_id"],
            "severity": spec["severity"],
            "source_type": source_type_for_rule(spec["rule_id"]),
            "feature_column": f"rule_{spec['rule_id']}_flag",
        }
        for spec in rule_specs
    ])


def add_rule_flags(df: DataFrame, rule_specs: list[dict]) -> DataFrame:
    for spec in rule_specs:
        df = df.withColumn(f"rule_{spec['rule_id']}_flag", flag(spec["condition"]))

    def sum_cols(cols):
        if not cols:
            return F.lit(0.0)
        return reduce(lambda a, b: a + b, [F.col(c) for c in cols])

    critical_cols = [f"rule_{s['rule_id']}_flag" for s in rule_specs if s["severity"] == "critical"]
    high_cols = [f"rule_{s['rule_id']}_flag" for s in rule_specs if s["severity"] == "high"]
    medium_cols = [f"rule_{s['rule_id']}_flag" for s in rule_specs if s["severity"] == "medium"]
    low_cols = [f"rule_{s['rule_id']}_flag" for s in rule_specs if s["severity"] == "low"]
    all_rule_cols = critical_cols + high_cols + medium_cols + low_cols

    return (
        df
        .withColumn("critical_rule_fail_count", sum_cols(critical_cols))
        .withColumn("high_rule_fail_count", sum_cols(high_cols))
        .withColumn("medium_rule_fail_count", sum_cols(medium_cols))
        .withColumn("low_rule_fail_count", sum_cols(low_cols))
        .withColumn("total_rule_fail_count", sum_cols(all_rule_cols))
        .withColumn("has_rule_failure_flag", flag(F.col("total_rule_fail_count") > 0))
        .withColumn(
            "max_rule_severity_rank",
            F.when(F.col("critical_rule_fail_count") > 0, F.lit(4.0))
            .when(F.col("high_rule_fail_count") > 0, F.lit(3.0))
            .when(F.col("medium_rule_fail_count") > 0, F.lit(2.0))
            .when(F.col("low_rule_fail_count") > 0, F.lit(1.0))
            .otherwise(F.lit(0.0)),
        )
        .withColumn("rule_metadata_json", F.lit(rule_metadata(rule_specs)))
    )


class AutoEncoder(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int):
        super().__init__()
        bottleneck_dim = max(2, hidden_dim // 2)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, bottleneck_dim),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


def load_model(model_state_b64: str, input_dim: int, hidden_dim: int) -> AutoEncoder:
    model = AutoEncoder(input_dim=input_dim, hidden_dim=hidden_dim)
    raw = base64.b64decode(model_state_b64.encode("ascii"))
    try:
        state = torch.load(BytesIO(raw), map_location="cpu", weights_only=True)
    except TypeError:
        state = torch.load(BytesIO(raw), map_location="cpu")
    model.load_state_dict(state)
    model.eval()
    return model


def load_latest_artifact(layer: str):
    artifacts_df = spark.table(MODEL_ARTIFACTS_TABLE).where(F.col("layer") == layer)
    if "granularity" in artifacts_df.columns:
        artifacts_df = artifacts_df.where(F.col("granularity") == "record")
    if MODEL_VERSION_WIDGET:
        artifacts_df = artifacts_df.where(F.col("model_version") == MODEL_VERSION_WIDGET)
    artifact_pdf = (
        artifacts_df
        .orderBy(F.col("trained_at").desc_nulls_last(), F.col("model_version").desc())
        .limit(1)
        .toPandas()
    )
    if artifact_pdf.empty:
        raise ValueError(f"No record-level AIOps model artifact found for layer={layer}. Run 02_aiops_train_autoencoder first.")
    return artifact_pdf.iloc[0].to_dict()


def contribution_json(feature_cols: list[str], contributions: np.ndarray) -> tuple[str, str, str]:
    top_idx = np.argsort(contributions)[::-1][:TOP_N_FEATURES]
    top_rows = [
        {
            "feature": feature_cols[int(i)],
            "contribution": float(contributions[int(i)]),
            "source_type": feature_source_type(feature_cols[int(i)]),
        }
        for i in top_idx
    ]
    coverage = {}
    for name, value in zip(feature_cols, contributions):
        source_type = feature_source_type(name)
        coverage[source_type] = coverage.get(source_type, 0.0) + float(value)
    primary_source_type = max(coverage.items(), key=lambda kv: kv[1])[0] if coverage else "unknown"
    return json.dumps(top_rows), json.dumps(coverage), primary_source_type


def prepare_for_model(features_pdf: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    for c in feature_cols:
        if c not in features_pdf.columns:
            features_pdf[c] = 0.0
        features_pdf[c] = pd.to_numeric(features_pdf[c], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return features_pdf


def score_features(features_df: DataFrame, layer: str, batch_id: int, artifact: dict) -> DataFrame:
    feature_cols = json.loads(artifact["feature_columns_json"])
    features_pdf = features_df.orderBy("event_ts", "record_id").toPandas()
    if features_pdf.empty:
        return None
    features_pdf = prepare_for_model(features_pdf, feature_cols)

    mean = np.array(json.loads(artifact["mean_json"]), dtype="float32")
    std = np.array(json.loads(artifact["std_json"]), dtype="float32")
    std[std == 0] = 1.0
    X_raw = features_pdf[feature_cols].astype("float32").to_numpy()
    X = (X_raw - mean) / std

    model = load_model(artifact["model_state_b64"], input_dim=len(feature_cols), hidden_dim=int(artifact["hidden_dim"]))
    with torch.no_grad():
        reconstructed = model(torch.tensor(X, dtype=torch.float32)).numpy()

    contributions = (X - reconstructed) ** 2
    errors = np.mean(contributions, axis=1)
    anomaly_threshold = float(artifact["anomaly_threshold"])
    warning_threshold = anomaly_threshold * WARNING_MULTIPLIER
    quarantine_threshold = anomaly_threshold * QUARANTINE_MULTIPLIER

    rows = []
    for idx, error in enumerate(errors):
        top_features_json, coverage_json, primary_source_type = contribution_json(feature_cols, contributions[idx])
        if float(error) > quarantine_threshold:
            decision_action = "QUARANTINE"
            severity = "CRITICAL"
            validated = False
        elif float(error) > warning_threshold:
            decision_action = "WARN"
            severity = "WARNING"
            validated = True
        else:
            decision_action = "PASS"
            severity = "NORMAL"
            validated = True

        ingest_value = features_pdf.iloc[idx].get("ingest_date")
        if pd.isna(ingest_value):
            ingest_value = features_pdf.iloc[idx].get("feature_date")

        rows.append({
            "batch_id": int(batch_id),
            "layer": layer,
            "record_id": str(features_pdf.iloc[idx]["record_id"]),
            "business_key": None if pd.isna(features_pdf.iloc[idx].get("business_key")) else str(features_pdf.iloc[idx].get("business_key")),
            "event_ts": None if pd.isna(features_pdf.iloc[idx]["event_ts"]) else str(features_pdf.iloc[idx]["event_ts"]),
            "feature_date": None if pd.isna(features_pdf.iloc[idx]["feature_date"]) else str(features_pdf.iloc[idx]["feature_date"]),
            "ingest_date": None if pd.isna(ingest_value) else str(ingest_value),
            "model_name": str(artifact.get("model_name", "pytorch_record_autoencoder")),
            "model_version": str(artifact["model_version"]),
            "granularity": "record",
            "reconstruction_error": float(error),
            "anomaly_threshold": anomaly_threshold,
            "warning_threshold": float(warning_threshold),
            "quarantine_threshold": float(quarantine_threshold),
            "decision_action": decision_action,
            "aiops_severity": severity,
            "validated_for_next_layer": bool(validated),
            "primary_source_type": primary_source_type,
            "source_type_coverage_json": coverage_json,
            "top_contributing_features_json": top_features_json,
        })

    score_schema = T.StructType([
        T.StructField("batch_id", T.LongType(), False),
        T.StructField("layer", T.StringType(), False),
        T.StructField("record_id", T.StringType(), False),
        T.StructField("business_key", T.StringType(), True),
        T.StructField("event_ts", T.StringType(), True),
        T.StructField("feature_date", T.StringType(), True),
        T.StructField("ingest_date", T.StringType(), True),
        T.StructField("model_name", T.StringType(), False),
        T.StructField("model_version", T.StringType(), False),
        T.StructField("granularity", T.StringType(), False),
        T.StructField("reconstruction_error", T.DoubleType(), False),
        T.StructField("anomaly_threshold", T.DoubleType(), False),
        T.StructField("warning_threshold", T.DoubleType(), False),
        T.StructField("quarantine_threshold", T.DoubleType(), False),
        T.StructField("decision_action", T.StringType(), False),
        T.StructField("aiops_severity", T.StringType(), False),
        T.StructField("validated_for_next_layer", T.BooleanType(), False),
        T.StructField("primary_source_type", T.StringType(), True),
        T.StructField("source_type_coverage_json", T.StringType(), True),
        T.StructField("top_contributing_features_json", T.StringType(), True),
    ])

    return (
        spark.createDataFrame(rows, schema=score_schema)
        .withColumn("event_ts", F.to_timestamp("event_ts"))
        .withColumn("feature_date", F.to_date("feature_date"))
        .withColumn("ingest_date", F.to_date("ingest_date"))
        .withColumn("scored_at", F.current_timestamp())
        .withColumn("scored_date", F.to_date("scored_at"))
    )


def register_delta_table(path: str, table_name: str):
    spark.sql(f"CREATE TABLE IF NOT EXISTS {table_name} USING DELTA LOCATION '{path}'")
    spark.sql(f"REFRESH TABLE {table_name}")

In [ ]:
def add_record_keys(df: DataFrame) -> DataFrame:
    return (
        df
        .withColumn("record_id", F.concat_ws("|", F.coalesce(F.col("topic"), F.lit("")), F.col("partition").cast("string"), F.col("offset").cast("string")))
        .withColumn("business_key", F.lit(None).cast("string"))
        .withColumn("event_ts_for_aiops", F.col("bronze_ingest_ts"))
        .withColumn("feature_date_for_aiops", F.to_date("bronze_ingest_ts"))
    )


def build_record_features(batch_df: DataFrame) -> DataFrame:
    duplicate_w = Window.partitionBy("partition", "offset")
    base = (
        add_record_keys(batch_df)
        .withColumn("layer", F.lit("bronze"))
        .withColumn("event_ts", F.col("event_ts_for_aiops"))
        .withColumn("feature_date", F.col("feature_date_for_aiops"))
        .withColumn("source_table", F.lit(SOURCE_TABLE))
        .withColumn("duplicate_partition_offset_count", F.count("*").over(duplicate_w).cast("double"))
        .withColumn("enqueue_to_bronze_delay_sec", (F.col("bronze_ingest_ts").cast("long") - F.col("eventhub_enqueued_ts").cast("long")).cast("double"))
    )
    rule_specs = [
        {"rule_id": "critical_topic_null", "severity": "critical", "condition": F.col("topic").isNull()},
        {"rule_id": "critical_partition_null", "severity": "critical", "condition": F.col("partition").isNull()},
        {"rule_id": "critical_offset_null", "severity": "critical", "condition": F.col("offset").isNull()},
        {"rule_id": "critical_eventhub_enqueued_ts_null", "severity": "critical", "condition": F.col("eventhub_enqueued_ts").isNull()},
        {"rule_id": "critical_raw_json_null", "severity": "critical", "condition": F.col("raw_json").isNull()},
        {"rule_id": "critical_bronze_ingest_ts_null", "severity": "critical", "condition": F.col("bronze_ingest_ts").isNull()},
        {"rule_id": "critical_parse_ok_false", "severity": "critical", "condition": F.coalesce(F.col("parse_ok"), F.lit(False)) == F.lit(False)},
        {"rule_id": "critical_duplicate_partition_offset", "severity": "critical", "condition": F.col("duplicate_partition_offset_count") > 1},
        {"rule_id": "high_parse_error_present", "severity": "high", "condition": F.col("parse_error").isNotNull()},
        {"rule_id": "high_source_not_hsl_hfp_mqtt", "severity": "high", "condition": F.coalesce(F.col("source"), F.lit("")) != F.lit("hsl_hfp_mqtt")},
        {"rule_id": "high_event_type_not_vp", "severity": "high", "condition": F.coalesce(F.col("event_type"), F.lit("")) != F.lit("vp")},
        {"rule_id": "high_mqtt_topic_null", "severity": "high", "condition": F.col("mqtt_topic").isNull()},
        {"rule_id": "high_transport_mode_null", "severity": "high", "condition": F.col("transport_mode").isNull()},
        {"rule_id": "medium_producer_ts_bad_format", "severity": "medium", "condition": F.col("producer_ingest_ts_utc").isNotNull() & (~F.col("producer_ingest_ts_utc").rlike(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(?:\.\d+)?Z$"))},
    ]
    return add_rule_flags(base, rule_specs)

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")

SOURCE_TABLE = qname(SOURCE_TABLE_NAME)
DECISION_TABLE = qname(DECISION_TABLE_NAME)
SCORE_TABLE = qname(SCORE_TABLE_NAME)
VALIDATED_TABLE = qname(VALIDATED_TABLE_NAME)
WARNING_TABLE = qname(WARNING_TABLE_NAME)
QUARANTINE_TABLE = qname(QUARANTINE_TABLE_NAME)

DECISION_PATH = f"{AIOPS_BASE_PATH}/{LAYER}/decisions"
SCORE_PATH = f"{AIOPS_BASE_PATH}/{LAYER}/scores"
VALIDATED_PATH = f"{AIOPS_BASE_PATH}/{LAYER}/validated"
WARNING_PATH = f"{AIOPS_BASE_PATH}/{LAYER}/warning"
QUARANTINE_PATH = f"{AIOPS_BASE_PATH}/{LAYER}/quarantine"
CHECKPOINT_PATH = f"{AIOPS_BASE_PATH}/{LAYER}/checkpoints/score_records"

artifact = load_latest_artifact(LAYER)
print(json.dumps({
    "layer": LAYER,
    "source_table": SOURCE_TABLE,
    "model_version": artifact["model_version"],
    "feature_count": len(json.loads(artifact["feature_columns_json"])),
    "source_path": SOURCE_TABLE,
}, indent=2))


def delta_path_exists(path: str) -> bool:
    try:
        spark.read.format("delta").load(path).limit(1).collect()
        return True
    except Exception:
        return False


def empty_decisions_df() -> DataFrame:
    schema = T.StructType([
        T.StructField("batch_id", T.LongType(), True),
        T.StructField("layer", T.StringType(), True),
        T.StructField("record_id", T.StringType(), True),
        T.StructField("business_key", T.StringType(), True),
        T.StructField("event_ts", T.TimestampType(), True),
        T.StructField("feature_date", T.DateType(), True),
        T.StructField("ingest_date", T.DateType(), True),
        T.StructField("model_name", T.StringType(), True),
        T.StructField("model_version", T.StringType(), True),
        T.StructField("granularity", T.StringType(), True),
        T.StructField("reconstruction_error", T.DoubleType(), True),
        T.StructField("anomaly_threshold", T.DoubleType(), True),
        T.StructField("warning_threshold", T.DoubleType(), True),
        T.StructField("quarantine_threshold", T.DoubleType(), True),
        T.StructField("decision_action", T.StringType(), True),
        T.StructField("aiops_severity", T.StringType(), True),
        T.StructField("validated_for_next_layer", T.BooleanType(), True),
        T.StructField("primary_source_type", T.StringType(), True),
        T.StructField("source_type_coverage_json", T.StringType(), True),
        T.StructField("top_contributing_features_json", T.StringType(), True),
        T.StructField("scored_at", T.TimestampType(), True),
        T.StructField("scored_date", T.DateType(), True),
    ])
    return spark.createDataFrame([], schema=schema)


def empty_enriched_records_df() -> DataFrame:
    decision_cols_for_records = [
        "batch_id",
        "model_name", "model_version", "granularity", "reconstruction_error",
        "anomaly_threshold", "warning_threshold", "quarantine_threshold",
        "decision_action", "aiops_severity", "validated_for_next_layer",
        "primary_source_type", "source_type_coverage_json",
        "top_contributing_features_json", "scored_at", "scored_date",
    ]
    df = add_record_keys(spark.table(SOURCE_TABLE).limit(0))
    for col_name in decision_cols_for_records:
        field = empty_decisions_df().schema[col_name]
        df = df.withColumn(col_name, F.lit(None).cast(field.dataType))
    return df


def precreate_delta_output(path: str, table_name: str, df: DataFrame, partition_cols: list[str] | None = None) -> None:
    if not delta_path_exists(path):
        writer = (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
        )
        if partition_cols:
            writer = writer.partitionBy(*partition_cols)
        writer.save(path)
    register_delta_table(path, table_name)


def precreate_output_tables() -> None:
    if not PERSIST_REPORTS:
        return
    decisions_empty = empty_decisions_df()
    records_empty = empty_enriched_records_df()
    precreate_delta_output(DECISION_PATH, DECISION_TABLE, decisions_empty, ["ingest_date", "layer"])
    precreate_delta_output(SCORE_PATH, SCORE_TABLE, decisions_empty, ["ingest_date", "layer"])
    precreate_delta_output(VALIDATED_PATH, VALIDATED_TABLE, records_empty, ["ingest_date"])
    precreate_delta_output(WARNING_PATH, WARNING_TABLE, records_empty, ["ingest_date"])
    precreate_delta_output(QUARANTINE_PATH, QUARANTINE_TABLE, records_empty, ["ingest_date"])


precreate_output_tables()


def write_runtime_metric(batch_id: int, input_record_count: int, scored_record_count: int, started: float, notes: str) -> None:
    if not PERSIST_REPORTS:
        return
    runtime_df = spark.createDataFrame([{
        "layer": LAYER,
        "runtime_stage": "aiops_record_score",
        "batch_id": int(batch_id),
        "input_record_count": int(input_record_count),
        "scored_record_count": int(scored_record_count),
        "runtime_ms": float((time.perf_counter() - started) * 1000.0),
        "model_version": str(artifact["model_version"]),
        "notes": notes,
    }]).withColumn("runtime_ts", F.current_timestamp()).withColumn("runtime_date", F.to_date("runtime_ts"))
    runtime_df.write.format("delta").mode("append").option("mergeSchema", "true").partitionBy("runtime_date", "layer").save(RUNTIME_PATH)
    register_delta_table(RUNTIME_PATH, RUNTIME_TABLE)


CURRENT_LAYER_DECISION_RECORD_COLUMNS = [
    "batch_id",
    "model_name", "model_version", "granularity", "reconstruction_error",
    "anomaly_threshold", "warning_threshold", "quarantine_threshold",
    "decision_action", "aiops_severity", "validated_for_next_layer",
    "primary_source_type", "source_type_coverage_json",
    "top_contributing_features_json", "scored_at", "scored_date",
]


def drop_existing_decision_columns(df: DataFrame) -> DataFrame:
    existing_cols = [c for c in CURRENT_LAYER_DECISION_RECORD_COLUMNS if c in df.columns]
    return df.drop(*existing_cols) if existing_cols else df


def write_scored_batch(batch_df: DataFrame, batch_id: int) -> None:
    started = time.perf_counter()
    batch_df = filter_to_target_ingest_date(batch_df)
    if batch_df.isEmpty():
        return
    cached_batch_df = batch_df.persist()
    features_ranked_df = None
    total_scored = 0

    try:
        input_record_count = int(cached_batch_df.count())
        if input_record_count == 0:
            return

        feature_cols = json.loads(artifact["feature_columns_json"])
        missing_source_cols = [c for c in feature_cols if c not in build_record_features(cached_batch_df).columns]
        if missing_source_cols:
            raise ValueError(f"Model expects missing Bronze feature columns: {missing_source_cols}")

        keyed_records_df = drop_existing_decision_columns(add_record_keys(cached_batch_df))
        features_ranked_df = (
            build_record_features(cached_batch_df)
            .withColumn(
                "_score_rank",
                F.row_number().over(Window.orderBy(F.col("event_ts").asc_nulls_last(), F.col("record_id").asc_nulls_last()))
            )
            .persist()
        )

        chunk_size = min(MAX_DRIVER_RECORDS_PER_BATCH, SCORING_CHUNK_RECORDS)
        if chunk_size <= 0:
            chunk_size = MAX_DRIVER_RECORDS_PER_BATCH

        for chunk_start in range(1, input_record_count + 1, chunk_size):
            chunk_end = min(chunk_start + chunk_size - 1, input_record_count)
            chunk_features_df = (
                features_ranked_df
                .where((F.col("_score_rank") >= F.lit(chunk_start)) & (F.col("_score_rank") <= F.lit(chunk_end)))
                .drop("_score_rank")
            )

            decisions_df = score_features(chunk_features_df, LAYER, batch_id, artifact)
            if decisions_df is None:
                continue

            decisions_df = decisions_df.persist()
            try:
                scored_chunk_count = int(decisions_df.count())
                total_scored += scored_chunk_count

                decision_cols_for_records = [
                    "record_id", "batch_id",
                    "model_name", "model_version", "granularity", "reconstruction_error",
                    "anomaly_threshold", "warning_threshold", "quarantine_threshold",
                    "decision_action", "aiops_severity", "validated_for_next_layer",
                    "primary_source_type", "source_type_coverage_json",
                    "top_contributing_features_json", "scored_at", "scored_date",
                ]

                enriched_records_df = (
                    keyed_records_df
                    .join(decisions_df.select(*decision_cols_for_records), on="record_id", how="inner")
                )

                if PERSIST_REPORTS:
                    decisions_df.write.format("delta").mode("append").option("mergeSchema", "true").partitionBy("ingest_date", "layer").save(DECISION_PATH)
                    decisions_df.write.format("delta").mode("append").option("mergeSchema", "true").partitionBy("ingest_date", "layer").save(SCORE_PATH)
                    enriched_records_df.where(F.col("validated_for_next_layer")).write.format("delta").mode("append").option("mergeSchema", "true").partitionBy("ingest_date").save(VALIDATED_PATH)
                    enriched_records_df.where(F.col("decision_action") == "WARN").write.format("delta").mode("append").option("mergeSchema", "true").partitionBy("ingest_date").save(WARNING_PATH)
                    enriched_records_df.where(F.col("decision_action") == "QUARANTINE").write.format("delta").mode("append").option("mergeSchema", "true").partitionBy("ingest_date").save(QUARANTINE_PATH)
            finally:
                decisions_df.unpersist()

        if PERSIST_REPORTS:
            register_delta_table(DECISION_PATH, DECISION_TABLE)
            register_delta_table(SCORE_PATH, SCORE_TABLE)
            register_delta_table(VALIDATED_PATH, VALIDATED_TABLE)
            register_delta_table(WARNING_PATH, WARNING_TABLE)
            register_delta_table(QUARANTINE_PATH, QUARANTINE_TABLE)

        write_runtime_metric(
            batch_id=batch_id,
            input_record_count=input_record_count,
            scored_record_count=total_scored,
            started=started,
            notes=f"chunked_scoring_chunk_size={chunk_size}",
        )
    finally:
        if features_ranked_df is not None:
            features_ranked_df.unpersist()
        cached_batch_df.unpersist()


for q in spark.streams.active:
    if q.name == QUERY_NAME:
        q.stop()

stream_reader = spark.readStream
if MAX_FILES_PER_TRIGGER > 0:
    stream_reader = stream_reader.option("maxFilesPerTrigger", MAX_FILES_PER_TRIGGER)
if MAX_BYTES_PER_TRIGGER:
    stream_reader = stream_reader.option("maxBytesPerTrigger", MAX_BYTES_PER_TRIGGER)

score_query = (
    stream_reader.table(SOURCE_TABLE)
    .writeStream
    .foreachBatch(write_scored_batch)
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .queryName(QUERY_NAME)
    .trigger(processingTime=TRIGGER_INTERVAL)
    .start()
)

print(f"Started {QUERY_NAME}: {score_query.id}")
score_query.awaitTermination()
